# M2AD label-unaware su PVGIS (2005–2019)

Launcher riproducibile per l'implementazione di Alnegheimish et al. (AISTATS 2025). Ogni località è un asset indipendente; il training usa 2005–2018 e il 2019 resta completamente held-out. Le label non entrano mai nel modello o nella calibrazione. La run esporta inoltre le decisioni orarie 2016–2018 e 2019 nel contratto CSV consumato dai branch SDE.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PVGIS_DIR = Path('/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance')
OUT_DIR = REPO_ROOT / 'outputs' / 'pvgis_m2ad_2005_2019'
RUN = True
ALLOW_OVERWRITE = False

In [ ]:
command = [
    sys.executable, '-m', 'physiq_pv.experiments.pvgis_m2ad_runner',
    '--pvgis-dir', str(PVGIS_DIR),
    '--train-years', '2005-2018', '--export-train-years', '2016-2018',
    '--test-year', '2019',
    '--window-size', '120', '--error', 'area', '--area-half-window', '2',
    '--epochs', '30', '--gmm-components', 'bic', '--max-components', '3',
    '--significance', '0.001', '--device', 'cuda',
    '--out-dir', str(OUT_DIR),
]
print(shlex.join(command))

In [ ]:
if RUN:
    if not PVGIS_DIR.is_dir():
        raise NotADirectoryError(f'Directory PVGIS non trovata: {PVGIS_DIR}')
    if OUT_DIR.exists() and any(OUT_DIR.iterdir()) and not ALLOW_OVERWRITE:
        raise FileExistsError(
            f'Output non vuoto: {OUT_DIR}. Scegliere una nuova directory o impostare '
            'ALLOW_OVERWRITE=True esplicitamente.'
        )
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print('Dry run: imposta RUN=True per avviare il training completo.')

In [ ]:
import pandas as pd

if (OUT_DIR / 'm2ad_location_summary.csv').exists():
    display(pd.read_csv(OUT_DIR / 'm2ad_location_summary.csv'))
if (OUT_DIR / 'm2ad_intervals.csv').exists():
    display(pd.read_csv(OUT_DIR / 'm2ad_intervals.csv').head(20))
for name in ('train_anomaly_scores.csv', 'anomaly_scores.csv'):
    path = OUT_DIR / name
    if path.exists():
        print(name, pd.read_csv(path, nrows=5).columns.tolist())